In [1]:
%pip install openpyxl

In [5]:
import pandas as pd
import numpy as np
import glob

# Commandes (fichiers disponibles)
orders_files = sorted(glob.glob("data/orders_2024_*.csv"))
df_orders = pd.concat(
    [pd.read_csv(f, parse_dates=["order_datetime"]) for f in orders_files],
    ignore_index=True
)

# Produits et catégories (deux feuilles du même fichier Excel)
sheets = pd.read_excel("data/products.xlsx", sheet_name=["products", "categories"])
df_products = sheets["products"]
df_categories = sheets["categories"]

print(df_orders.shape)
print(df_products.head())
print(df_categories.head())
print(df_products.columns.tolist())
print(df_categories.columns.tolist())

(10000, 7)
   product_id product_name  category_id category_name  weight_kg
0           1    Product 1            1   Electronics      11.55
1           2    Product 2            3         Sport       2.22
2           3    Product 3            5          Toys      11.43
3           4    Product 4            1   Electronics       4.17
4           5    Product 5            5          Toys       1.81
   category_id category_name                  description
0            1   Electronics  Description for Electronics
1            2          Home         Description for Home
2            3         Sport        Description for Sport
3            4       Fashion      Description for Fashion
4            5          Toys         Description for Toys
['product_id', 'product_name', 'category_id', 'category_name', 'weight_kg']
['category_id', 'category_name', 'description']


In [3]:
import os
print(os.listdir("data"))

['orders_2024_01.csv', 'orders_2024_03.csv', 'iris.csv', 'matplotlib.png', 'fasta-example.fasta', 'Museums_in_DC.geojson', 'marie.png', 'bar.vl.json', 'audio.wav']


In [4]:
import os
print(os.listdir("data"))

['orders_2024_01.csv', 'orders_2024_03.csv', 'products.xlsx', 'iris.csv', 'matplotlib.png', 'fasta-example.fasta', 'Museums_in_DC.geojson', 'marie.png', 'bar.vl.json', 'audio.wav']


In [6]:
df_prod_full = df_products.merge(
    df_categories,
    on="category_id",
    how="left",
    suffixes=("", "_cat")
)

print(df_prod_full.head())
print(df_prod_full.columns.tolist())

   product_id product_name  category_id category_name  weight_kg  \
0           1    Product 1            1   Electronics      11.55   
1           2    Product 2            3         Sport       2.22   
2           3    Product 3            5          Toys      11.43   
3           4    Product 4            1   Electronics       4.17   
4           5    Product 5            5          Toys       1.81   

  category_name_cat                  description  
0       Electronics  Description for Electronics  
1             Sport        Description for Sport  
2              Toys         Description for Toys  
3       Electronics  Description for Electronics  
4              Toys         Description for Toys  
['product_id', 'product_name', 'category_id', 'category_name', 'weight_kg', 'category_name_cat', 'description']


In [7]:
df_orders_prod = df_orders.merge(
    df_prod_full,
    on="product_id",
    how="left"
)

print(df_orders_prod[["order_id", "product_id", "product_name", "category_id", "category_name"]].head())
print(df_orders_prod.shape)

   order_id  product_id product_name  category_id category_name
0         1          26   Product 26            3         Sport
1         2          16   Product 16            3         Sport
2         3          25   Product 25            3         Sport
3         4           9    Product 9            5          Toys
4         5          24   Product 24            1   Electronics
(10000, 13)


In [8]:
print("Nombre de lignes avant/après :", len(df_orders), len(df_orders_prod))
print("Commandes sans catégorie :", df_orders_prod["category_name"].isna().sum())

Nombre de lignes avant/après : 10000 10000
Commandes sans catégorie : 0


In [9]:
df_orders_prod["revenue"] = df_orders_prod["price"] * df_orders_prod["quantity"]

category_metrics = (
    df_orders_prod
    .groupby("category_name")
    .agg(
        revenue_sum=("revenue", "sum"),
        orders_count=("order_id", "nunique"),
        items_count=("quantity", "sum"),
    )
    .sort_values("revenue_sum", ascending=False)
)

print(category_metrics)

               revenue_sum  orders_count  items_count
category_name                                        
Toys           78184841.98          3027         7544
Sport          68066220.13          2614         6507
Electronics    51795912.29          1976         5054
Fashion        35236511.06          1365         3455
Home           25892222.82          1018         2550


In [10]:
top5_categories = category_metrics.head(5)
print(top5_categories)

               revenue_sum  orders_count  items_count
category_name                                        
Toys           78184841.98          3027         7544
Sport          68066220.13          2614         6507
Electronics    51795912.29          1976         5054
Fashion        35236511.06          1365         3455
Home           25892222.82          1018         2550


In [11]:
print(category_metrics["revenue_sum"].sum())
print(df_orders_prod["revenue"].sum())

259175708.28
259175708.28


In [12]:
user_cat = (
    df_orders_prod
    .groupby(["user_id", "category_name"], as_index=False)["revenue"]
    .sum()
)

user_total = (
    df_orders_prod
    .groupby("user_id")["revenue"]
    .sum()
    .rename("revenue_sum")
    .reset_index()
)

print(user_cat.head())
print(user_total.head())

   user_id category_name    revenue
0     1001   Electronics  806709.73
1     1001       Fashion  157895.57
2     1001          Home  366697.85
3     1001         Sport  930587.11
4     1001          Toys  917896.33
   user_id  revenue_sum
0     1001   3179786.59
1     1002   1935678.62
2     1003   2868242.39
3     1004   2566296.37
4     1005   1950253.65


In [13]:
top_cat = (
    user_cat
    .sort_values(["user_id", "revenue"], ascending=[True, False])
    .drop_duplicates("user_id")
    .rename(columns={"category_name": "top_category",
                     "revenue": "top_category_revenue"})
)

print(top_cat.head())

    user_id top_category  top_category_revenue
3      1001        Sport             930587.11
9      1002         Toys             613414.00
14     1003         Toys            1097690.91
18     1004        Sport             773548.45
24     1005         Toys             699113.40


In [14]:
user_features = top_cat.merge(user_total, on="user_id", how="left")

user_features["top_category_share"] = (
    user_features["top_category_revenue"] / user_features["revenue_sum"]
)

print(user_features.head())
print(user_features.shape)
print(user_features["top_category_share"].describe())

   user_id top_category  top_category_revenue  revenue_sum  top_category_share
0     1001        Sport             930587.11   3179786.59            0.292657
1     1002         Toys             613414.00   1935678.62            0.316899
2     1003         Toys            1097690.91   2868242.39            0.382705
3     1004        Sport             773548.45   2566296.37            0.301426
4     1005         Toys             699113.40   1950253.65            0.358473
(100, 5)
count    100.000000
mean       0.327486
std        0.045152
min        0.243095
25%        0.293289
50%        0.321115
75%        0.357027
max        0.470570
Name: top_category_share, dtype: float64
